![image_1779898861239.png](./image_1779898861239.png "image_1779898861239.png")

![image_1779898889699.png](./image_1779898889699.png "image_1779898889699.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql import functions as f

# Initialize Spark
spark = SparkSession.builder.appName("CustomersOrdersDF").getOrCreate()

# -------------------------
# Customers DataFrame
# -------------------------
customers_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("country", StringType(), True)
])

customers_data = [
    (1, "Alice", "USA"),
    (2, "Bob", "UK"),
    (3, "Charlie", "Canada")
]

customers_df = spark.createDataFrame(customers_data, schema=customers_schema)

# -------------------------
# Orders DataFrame
# -------------------------
orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", IntegerType(), True)
])

orders_data = [
    (1, 1, "2020-06-10", 5, 25),
    (2, 1, "2020-07-15", 10, 20),
    (3, 2, "2020-06-20", 4, 30),
    (4, 2, "2020-07-05", 2, 10),
    (5, 3, "2020-06-01", 5, 50),
    (6, 3, "2020-07-20", 10, 15)
]

orders_df = spark.createDataFrame(orders_data, schema=orders_schema)

# Show DataFrames
customers_df.show()
orders_df.show()

In [0]:
# merged_df=(
#     customers_df.join(orders_df, customers_df.customer_id == orders_df.customer_id, "inner")
#     .filter((f.col("quantity") * f.col("price")) > 100)
#     .select(
#         customers_df.customer_id,
#         customers_df.name,
#         orders_df.order_date
#     )
# )
# order_month_6_df=(
#     merged_df
#     .filter(f.month("order_date")==6)
#     .select("customer_id", "name")
# )
# order_month_7_df=(
#     merged_df
#     .filter(f.month("order_date")==7)
#     .select("customer_id", "name")
# )
# display(order_month_6_df.intersect(order_month_7_df))

In [0]:
final_df = (
    customers_df.join(
        orders_df, customers_df.customer_id == orders_df.customer_id, "inner"
    )
    .filter(f.col("quantity") * f.col("price") > 100)
    .groupBy(customers_df.customer_id, customers_df.name)
    .agg(f.collect_set(f.month("order_date")).alias("months"))
    .filter((f.array_contains("months", 6)) & (f.array_contains("months", 7)))
    .select("customer_id", "name")
)
display(final_df)